In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. Transform: Convert images to tensors
transform = transforms.ToTensor()

# 2. Load dataset (MNIST for example)
train_dataset = datasets.MNIST(
    root="./data", train=True, transform=transform, download=True
)

# 3. Create DataLoader with batching
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,    # batch size
    shuffle=True      # shuffle each epoch
)

# 4. Iterate over DataLoader
for batch_idx, (images, labels) in enumerate(train_loader):
    print(f"Batch {batch_idx+1}:")
    print(f" - images shape: {images.shape}")   # (64, 1, 28, 28)
    print(f" - labels shape: {labels.shape}")   # (64,)
    
    # Example: forward pass with a model (dummy here)
    # outputs = model(images)
    
    # break after 1 batch for demo
    break


100.0%
100.0%
100.0%
100.0%


Batch 1:
 - images shape: torch.Size([64, 1, 28, 28])
 - labels shape: torch.Size([64])


In [2]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

# 1. Load iris data
iris = load_iris()
X, y = iris.data, iris.target

# 2. Normalize features (good practice)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 3. Convert to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# 4. Create a Dataset and DataLoader
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# 5. Iterate through batches
for batch_idx, (features, labels) in enumerate(loader):
    print(f"Batch {batch_idx+1}:")
    print(" Features shape:", features.shape)  # (16, 4)
    print(" Labels shape:", labels.shape)      # (16,)
    
    # Example forward step (dummy)
    # outputs = model(features)
    
    # Stop after 2 batches just for demo
    if batch_idx == 1:
        break


Batch 1:
 Features shape: torch.Size([16, 4])
 Labels shape: torch.Size([16])
Batch 2:
 Features shape: torch.Size([16, 4])
 Labels shape: torch.Size([16])


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# -------------------------
# 1. Load and preprocess data
# -------------------------
iris = load_iris()
X, y = iris.data, iris.target

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# Create dataset & dataloader
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# -------------------------
# 2. Define simple neural net
# -------------------------
class IrisNet(nn.Module):
    def __init__(self):
        super(IrisNet, self).__init__()
        self.fc1 = nn.Linear(4, 16)   # input features = 4
        self.fc2 = nn.Linear(16, 3)   # 3 classes
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = IrisNet()

# -------------------------
# 3. Training setup
# -------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
epochs = 50

# -------------------------
# 4. Training loop
# -------------------------
for epoch in range(epochs):
    for features, labels in loader:
        # Forward pass
        outputs = model(features)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# -------------------------
# 5. Evaluation
# -------------------------
with torch.no_grad():
    outputs = model(X_tensor)
    _, preds = torch.max(outputs, 1)

y_true = y
y_pred = preds.numpy()

# Metrics
acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
cm = confusion_matrix(y_true, y_pred)

print("\n🔎 Classification Results:")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")
print("\nConfusion Matrix:\n", cm)
